#Biodenoising - Animal vocalization denoising
This is a demo for animal vocalization denoising without access to clean data.
For the more info check the [associated page](https://mariusmiron.com/research/biodenoising/) and the code repository on [github](https://github.com/earthspecies/biodenoising).

First, let's install the package from pip:

In [ ]:
!pip install biodenoising

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.4/156.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 kB 30.8 MB/s eta 0:00:00

We import the libraries:

In [ ]:
from IPython import display as disp
import os
import torch
import torchaudio
from biodenoising import pretrained
from biodenoising.denoiser.dsp import convert_audio

We download some noisy animal vocalizations from the biodenoising_validation dataset. Note that these files, species, noise conditions were not seen during training, to test for generalization.

In [ ]:
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/12_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/14_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/36_underwater_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/30_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/34_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/3_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/21_terrestrial_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/52_underwater_original.wav
!wget https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/24_underwater_original.wav

--2024-10-07 08:43:02--  https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/12_terrestrial_original.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.12.207, 172.217.194.207, 172.253.118.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.12.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 166426 (163K) [audio/wav]
Saving to: ‘12_terrestrial_original.wav’

12_terrestrial_orig 100%[===================>] 162.53K   324KB/s    in 0.5s    

2024-10-07 08:43:03 (324 KB/s) - ‘12_terrestrial_original.wav’ saved [166426/166426]

--2024-10-07 08:43:03--  https://storage.googleapis.com/esp-public-files/biodenoising/demo/benchmark/14_terrestrial_original.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.12.207, 172.217.194.207, 172.253.118.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.12.207|:443... connected.
HTTP request sent, awaiting re

We set the device, gpu or cpu. You can use a computing instance with a GPU for faster processing.

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

Let's load the 16kHz model. If it's the first time you run this, it will download the model locally.

In [ ]:
model = pretrained.biodenoising16k_dns48().to(device)

Downloading: "https://storage.googleapis.com/esp-public-files/biodenoising/model-16kHz-dns48.th" to /root/.cache/torch/hub/checkpoints/model-16kHz-dns48.th
100%|██████████| 72.0M/72.0M [00:05<00:00, 14.6MB/s]


We use the model above to denoise the first demo sound.

In [ ]:
wav, sr = torchaudio.load(os.path.join('12_terrestrial_original.wav'))
wav = convert_audio(wav, sr, model.sample_rate, model.chin).to(device)
with torch.no_grad():
    denoised = model(wav[None])[0]
disp.display(disp.Audio(wav.data.cpu().numpy(), rate=model.sample_rate))
disp.display(disp.Audio(denoised.data.cpu().numpy(), rate=model.sample_rate))

We now process the remaining files.

In [ ]:
file_list = ['36_underwater_original.wav','14_terrestrial_original.wav','3_terrestrial_original.wav','21_terrestrial_original.wav','24_underwater_original.wav','30_terrestrial_original.wav','34_terrestrial_original.wav']
for f in file_list:
  wav, sr = torchaudio.load(os.path.join(f))
  wav = convert_audio(wav, sr, model.sample_rate, model.chin).to(device)
  with torch.no_grad():
      denoised = model(wav[None])[0]
  print(f)
  disp.display(disp.Audio(wav.data.cpu().numpy(), rate=model.sample_rate))
  disp.display(disp.Audio(denoised.data.cpu().numpy(), rate=model.sample_rate))

36_underwater_original.wav


14_terrestrial_original.wav


3_terrestrial_original.wav


21_terrestrial_original.wav


24_underwater_original.wav


30_terrestrial_original.wav


34_terrestrial_original.wav


You can download the whole benchmarking dataset from zenodo and process it (subfolder 16000/noisy). You can also check the clean vocalizations and the added noise.

In [ ]:
!wget https://zenodo.org/records/13736465/files/biodenoising_validation_1.0.zip
!unzip biodenoising_validation_1.0.zip

--2024-10-04 12:26:29--  https://zenodo.org/records/13736465/files/biodenoising_validation_1.0.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.79.172, 188.184.98.238, 188.184.103.159, ...
Connecting to zenodo.org (zenodo.org)|188.185.79.172|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 706219420 (674M) [application/octet-stream]
Saving to: ‘biodenoising_validation_1.0.zip?download=1’

biodenoising_valida 100%[===================>] 673.50M  29.1MB/s    in 24s     

2024-10-04 12:26:54 (27.8 MB/s) - ‘biodenoising_validation_1.0.zip?download=1’ saved [706219420/706219420]

